# Desafio Técnico - Gato Mestre (Ciência de Dados)
## Etapa 1: Obtenção, Mapeamento e Diagnóstico Exploratório dos Dados

Este notebook é responsável por:
1. Configurar o ambiente e importar bibliotecas de manipulação, visualização e auditoria de dados (`pandas`, `missingno`, `seaborn`, `itables`).
2. Definir os caminhos do projeto de forma reprodutível.
3. Documentar o Dicionário de Dados oficial e as regras de negócio de domínio do Cartola FC.
4. Definir as listas semânticas de colunas para direcionar a Análise Exploratória (EDA).
5. Carregar a base histórica de atletas (`base_case_gm.csv`) preservando seu estado bruto (**Opção B**).
6. Executar o diagnóstico de qualidade de dados (tipagem, nulos, valores únicos/frequências, duplicidades e matriz `missingno`), renderizados com tabelas dinâmicas interativas (`itables`).

### 1. Importação das Bibliotecas e Configurações

In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
import itables
from itables import init_notebook_mode, show

# Renderização inline do matplotlib
%matplotlib inline

# Configurações estéticas e de formatação
warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")

# Ativa tabelas interativas (com busca em tempo real, ordenação por coluna e paginação)
init_notebook_mode(all_interactive=True)
itables.options.maxBytes = 0
itables.options.classes = ["display", "nowrap"]
itables.options.lengthMenu = [10, 25, 50, 100]

print(f"Pandas version: {pd.__version__}")
print(f"Numpy version: {np.__version__}")
print(f"Itables version: {itables.__version__}")

### 2. Definição dos Caminhos do Projeto

In [ ]:
# Definição da raiz do projeto e pastas de dados
PROJECT_ROOT = Path("..").resolve()
DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
CSV_PATH = DATA_RAW_DIR / "base_case_gm.csv"

print(f"Diretório Raiz: {PROJECT_ROOT}")
print(f"Arquivo CSV: {CSV_PATH} (Existe: {CSV_PATH.exists()})")

### 3. Dicionário de Dados e Regras de Negócio

A granularidade da base é de **um registro por atleta e partida disputada**, cobrindo as temporadas de **2022 a 2025**.

> *Nota de Domínio*: Uma rodada normalmente reúne dez partidas, mas jogos podem ser remarcados para a rodada seguinte. Nesses casos, uma rodada concentra mais de dez partidas e um atleta pode figurar em mais de um registro dentro dela.

---

#### 3.1 Mapeamento por Domínio de Negócio

| Domínio | Colunas no CSV | Descrição | Tipo Esperado |
| :--- | :--- | :--- | :--- |
| **Identificação** | `atleta_id`, `apelido`, `ano`, `rodada_id`, `clube_id`, `posicao_id` | Chaves primárias, temporada, clube e função do atleta | Inteiros / Texto |
| **Situação do Atleta** | `status_pre`, `status_inicial`, `entrou_em_campo`, `minutos_jogados` | Status divulgado pré-rodada, escalação e tempo de jogo | Categórico / Booleano / Numérico |
| **Mercado (Cartola)** | `preco_num`, `variacao_num`, `media_num`, `jogos_num` | Variáveis financeiras e histórico acumulado | Float / Inteiro |
| **Desempenho (Target)** | `pontos_num` | Pontuação final obtida na rodada | Float |
| **Contexto do Confronto** | `home_dummy`, `opponent`, `match_id` | Mando de campo, adversário e chave de junção com a API | Binário (0/1) / Inteiro |
| **Scouts Positivos** | `G`, `A`, `SG`, `FF`, `FT`, `FD`, `DD`, `DP`, `DE`, `DS`, `FS`, `PS` | Contagens de ações ofensivas e defensivas positivas na rodada | Float/Int |
| **Scouts Negativos** | `GS`, `GC`, `CA`, `CV`, `FC`, `I`, `PP`, `PC` | Contagens de faltas, cartões, gols sofridos/contra e pênaltis | Float/Int |

---

#### 3.2 Detalhamento dos Campos e Domínios

##### Identificação
| Campo | Tipo | Descrição | Domínio |
| :--- | :--- | :--- | :--- |
| `atleta_id` | inteiro | Identificador do atleta | - |
| `apelido` | texto | Nome pelo qual o atleta é conhecido | - |
| `ano` | inteiro | Temporada | 2022 a 2025 |
| `rodada_id` | inteiro | Rodada do campeonato | 1 a 38 |
| `clube_id` | inteiro | Identificador do clube do atleta na rodada | consultável na API |
| `posicao_id` | inteiro | Posição do atleta | 1 a 6 |

##### Situação do Atleta
| Campo | Tipo | Descrição | Domínio |
| :--- | :--- | :--- | :--- |
| `status_pre` | texto | Situação divulgada antes da rodada | Provável, Dúvida, Nulo, Suspenso, Contundido |
| `status_inicial` | texto | Condição na escalação | titulares ou reservas |
| `entrou_em_campo` | booleano | Indicador de participação na partida | True / False |
| `minutos_jogados` | numérico | Minutos em campo na rodada | 0 a 90+ minutos de acréscimo |

##### Mercado
| Campo | Tipo | Descrição | Domínio |
| :--- | :--- | :--- | :--- |
| `preco_num` | numérico | Preço do atleta em cartoletas na rodada | sempre positivo |
| `variacao_num` | numérico | Variação do preço em relação à rodada anterior | pode ser negativa |
| `media_num` | numérico | Média de pontos do atleta na temporada, acumulada até a rodada | real |
| `jogos_num` | inteiro | Jogos disputados na temporada, acumulados até a rodada | $\ge 0$ |

##### Desempenho
| Campo | Tipo | Descrição | Domínio |
| :--- | :--- | :--- | :--- |
| `pontos_num` | numérico | Pontuação do atleta na rodada | pode ser negativa |

##### Contexto da Partida
| Campo | Tipo | Descrição | Domínio |
| :--- | :--- | :--- | :--- |
| `home_dummy` | inteiro | 1 quando o clube do atleta é mandante, 0 quando visitante | 0 ou 1 |
| `opponent` | inteiro | Identificador do clube adversário | consultável na API |
| `match_id` | inteiro | Identificador da partida - chave de ligação com a API | ID do jogo |

---

#### 3.3 Scouts da Rodada

##### Ações Positivas
| Campo | Descrição | Aplicação |
| :--- | :--- | :--- |
| `G` | Gol | Todas as posições |
| `A` | Assistência | Todas as posições |
| `SG` | Jogo sem sofrer gol | Goleiros e Defensores (Lateral/Zagueiro) |
| `FF` | Finalização para fora | Todas as posições |
| `FT` | Finalização na trave | Todas as posições |
| `FD` | Finalização defendida | Todas as posições |
| `DD` | Defesa difícil | Apenas Goleiros |
| `DP` | Defesa de pênalti | Apenas Goleiros |
| `DE` | Defesa (registro auxiliar de goleiro) | Apenas Goleiros |
| `DS` | Desarme | Todas as posições |
| `FS` | Falta sofrida | Todas as posições |
| `PS` | Pênalti sofrido | Todas as posições |

##### Ações Negativas
| Campo | Descrição | Aplicação |
| :--- | :--- | :--- |
| `GS` | Gol sofrido | Apenas Goleiros |
| `GC` | Gol contra | Todas as posições |
| `CA` | Cartão amarelo | Todas as posições |
| `CV` | Cartão vermelho | Todas as posições |
| `FC` | Falta cometida | Todas as posições |
| `I` | Impedimento | Todas as posições |
| `PP` | Pênalti perdido | Todas as posições |
| `PC` | Pênalti cometido | Todas as posições |

---

#### 3.4 Tabela de Posições

| `posicao_id` | Posição | Sigla | Scouts Específicos |
| :--- | :--- | :--- | :--- |
| **1** | Goleiro | gol | `DD`, `DP`, `DE`, `GS`, `SG` |
| **2** | Lateral | lat | `SG` |
| **3** | Zagueiro | zag | `SG` |
| **4** | Meia | mei | - |
| **5** | Atacante | ata | - |
| **6** | Técnico | tec | - |

---

#### 3.5 Observações e Regras Críticas de Domínio
- **Escopo dos Scouts**: Contagens exclusivas da própria rodada (não acumulados da temporada).
- **Campos Acumulados**: `media_num` e `jogos_num` são acumulados até a rodada atual pelo Cartola.
- **Variação de Preço**: `variacao_num` reflete a oscilação de patrimônio em decorrência da performance na rodada.
- **Especificidade de Scouts por Posição**: `DD`, `DP`, `DE` e `GS` aplicam-se apenas a goleiros (`posicao_id = 1`); `SG` aplica-se apenas a goleiros e defensores (`posicao_id in [1, 2, 3]`).

### 4. Definição das Listas Semânticas de Colunas

In [ ]:
# Agrupamentos de colunas por domínio semântico
COLS_ID = ["atleta_id", "apelido", "ano", "rodada_id", "clube_id", "posicao_id"]
COLS_SITUACAO = ["status_pre", "status_inicial", "entrou_em_campo", "minutos_jogados"]
COLS_MERCADO = ["preco_num", "variacao_num", "media_num", "jogos_num"]
COLS_TARGET = ["pontos_num"]
COLS_CONTEXTO = ["home_dummy", "opponent", "match_id"]
COLS_SCOUTS_POS = ["G", "A", "SG", "FF", "FT", "FD", "DD", "DP", "DE", "DS", "FS", "PS"]
COLS_SCOUTS_NEG = ["GS", "GC", "CA", "CV", "FC", "I", "PP", "PC"]
COLS_SCOUTS = COLS_SCOUTS_POS + COLS_SCOUTS_NEG

ALL_EXPECTED_COLS = COLS_ID + COLS_SITUACAO + COLS_MERCADO + COLS_TARGET + COLS_CONTEXTO + COLS_SCOUTS
print(f"Total de colunas mapeadas no dicionário: {len(ALL_EXPECTED_COLS)}")

### 5. Carga Inicial do Dataset Histórico (`base_case_gm.csv`) - Opção B (Dados Brutos)

In [ ]:
# Carga sem coerção forçada de tipos, preservando o estado bruto para mapeamento de inconsistências
df_raw = pd.read_csv(CSV_PATH)

print(f"Dimensões do dataset: {df_raw.shape[0]:,} linhas x {df_raw.shape[1]} colunas")
df_raw.head(5)

### 6. Diagnóstico Exploratório e Qualidade dos Dados

In [ ]:
def resumo_qualidade_dados(
    df: pd.DataFrame,
    colunas: list[str] | None = None,
    top_n: int = 4
) -> pd.DataFrame:
    """
    Executa uma auditoria detalhada de qualidade e integridade dos dados para um conjunto de variáveis.

    Esta função consolida metadados estruturais essenciais para responder à Questão 1 do desafio técnico
    ("Que inconsistências você encontrou na base?"), avaliando completude, tipagem inferida,
    cardinalidade de valores distintos e a distribuição percentual dos valores mais frequentes.

    Parâmetros:
    -----------
    df : pd.DataFrame
        DataFrame contendo a base de dados a ser auditada.
    colunas : list[str] | None, default=None
        Lista com os nomes das colunas a serem analisadas (ou grupos semânticos como COLS_ID, COLS_SCOUTS).
        Se None, todas as colunas presentes em `df` serão avaliadas.
    top_n : int, default=4
        Quantidade dos valores mais frequentes (incluindo NaN) a serem detalhados com sua respectiva proporção.

    Retorno:
    --------
    pd.DataFrame
        Tabela estruturada contendo:
        - coluna: Nome da variável.
        - tipo_dado: Tipo primitivo inferido pelo pandas (ex: int64, float64, object, bool).
        - total_linhas: Volume total de registros no dataset.
        - qtd_nulos: Quantidade absoluta de valores ausentes (NaN/None).
        - pct_nulos (%): Percentual relativo de ausências em relação ao total de linhas.
        - qtd_unicos: Cardinalidade de valores únicos não nulos (nunique).
        - pct_unicos (%): Proporção de valores distintos em relação às linhas válidas.
        - top_{top_n}_valores_frequentes: String descritiva com os N valores mais recorrentes e suas proporções.
    """
    cols = colunas if colunas is not None else df.columns.tolist()
    total_linhas = len(df)
    registros = []

    for col in cols:
        serie = df[col]
        n_nulos = int(serie.isna().sum())
        pct_nulos = (n_nulos / total_linhas) * 100
        n_validos = total_linhas - n_nulos
        n_unicos = int(serie.nunique(dropna=True))
        pct_unicos = (n_unicos / n_validos * 100) if n_validos > 0 else 0.0
        
        # Top N valores frequentes com proporção formatada
        top_vals_series = serie.value_counts(normalize=True, dropna=False).head(top_n)
        top_vals_str = ", ".join([
            f"{f'NaN' if pd.isna(val) else val} ({pct * 100:.1f}%)"
            for val, pct in top_vals_series.items()
        ])
        
        registros.append({
            "coluna": col,
            "tipo_dado": str(serie.dtype),
            "total_linhas": total_linhas,
            "qtd_nulos": n_nulos,
            "pct_nulos (%)": pct_nulos,
            "qtd_unicos": n_unicos,
            "pct_unicos (%)": pct_unicos,
            f"top_{top_n}_valores_frequentes": top_vals_str
        })
        
    df_resumo = pd.DataFrame(registros)
    return df_resumo


def analisar_duplicidades(
    df: pd.DataFrame,
    chaves_negocio: list[list[str]] | None = None
) -> dict:
    """
    Audita duplicidades estritas de linhas completas e avalia a unicidade de chaves primárias e de domínio.

    No contexto do Cartola FC, esta análise é fundamental para identificar:
    1. Registros 100% redundantes gerados por erros de ingestão.
    2. Partidas remarcadas (onde um mesmo atleta pode atuar mais de uma vez em uma mesma rodada do calendário).
    3. Violações de integridade relacional entre atleta e partida (`match_id`).

    Parâmetros:
    -----------
    df : pd.DataFrame
        DataFrame contendo a base histórica a ser inspecionada.
    chaves_negocio : list[list[str]] | None, default=None
        Lista de combinações de colunas que representam chaves lógicas de negócio.
        Por padrão, avalia:
        - ['atleta_id', 'ano', 'rodada_id'] (atleta por rodada do campeonato)
        - ['atleta_id', 'match_id'] (atleta por partida individual)
        - ['match_id', 'ano', 'rodada_id'] (partida por rodada)

    Retorno:
    --------
    dict
        Dicionário estruturado com volumetria e percentuais de duplicidade por chave avaliada.
    """
    if chaves_negocio is None:
        chaves_negocio = [
            ["atleta_id", "ano", "rodada_id"],           # Chave de rodada do atleta
            ["atleta_id", "match_id"],                  # Chave de partida do atleta
            ["match_id", "ano", "rodada_id"]            # Chave de partida na rodada
        ]
        
    dups_linhas_completas = int(df.duplicated().sum())
    resultado = {
        "total_linhas": len(df),
        "linhas_100pct_duplicadas": dups_linhas_completas,
        "duplicidades_por_chave": {}
    }
    
    for chave in chaves_negocio:
        if all(c in df.columns for c in chave):
            qtd_dups = int(df.duplicated(subset=chave, keep=False).sum())
            chave_nome = " + ".join(chave)
            resultado["duplicidades_por_chave"][chave_nome] = {
                "linhas_afetadas": qtd_dups,
                "pct_afetadas (%)": (qtd_dups / len(df)) * 100
            }
            
    return resultado


def plotar_diagnostico_nulos(
    df: pd.DataFrame,
    colunas: list[str] | None = None,
    figsize_matrix: tuple = (14, 5)
):
    """
    Gera diagnósticos visuais de dados faltantes: gráfico de barras quantitativo e matriz espectral (`missingno`).

    Visualizações geradas:
    1. Gráfico de Barras Horizontais: Quantifica o percentual exato de valores nulos para cada coluna com ausência.
    2. Matriz Espectral (missingno.matrix): Mapeia a localização espacial dos nulos ao longo do índice do dataset,
       permitindo identificar se os dados ausentes ocorrem de forma contígua (ex: bloco de anos específicos) ou aleatória.

    Parâmetros:
    -----------
    df : pd.DataFrame
        DataFrame contendo os dados a serem visualizados.
    colunas : list[str] | None, default=None
        Subconjunto opcional de colunas a serem plotadas. Se None, utiliza todas as colunas do dataset.
    figsize_matrix : tuple, default=(14, 5)
        Dimensões da figura para a matriz do missingno (largura, altura em polegadas).

    Retorno:
    --------
    None (renderiza as figuras diretamente no output do notebook).
    """
    cols = colunas if colunas is not None else df.columns.tolist()
    df_subset = df[cols]
    
    # 1. Gráfico de Barras com % de Nulos
    pct_nulos = df_subset.isna().mean() * 100
    cols_com_nulos = pct_nulos[pct_nulos > 0].sort_values(ascending=True)
    
    fig, ax = plt.subplots(figsize=(10, max(4, len(cols_com_nulos) * 0.35)))
    if len(cols_com_nulos) > 0:
        cols_com_nulos.plot(kind="barh", ax=ax, color="#d9534f", edgecolor="black", alpha=0.85)
        ax.set_title("Percentual de Valores Nulos por Coluna (%)", fontsize=13, pad=12)
        ax.set_xlabel("% de Nulos", fontsize=11)
        ax.set_ylabel("Coluna", fontsize=11)
        ax.set_xlim(0, max(100, cols_com_nulos.max() * 1.15))
        
        for p in ax.patches:
            width = p.get_width()
            ax.annotate(f"{width:.2f}%", (width + 1.0, p.get_y() + p.get_height() / 2.),
                        va="center", fontsize=10, color="#333333")
    else:
        ax.text(0.5, 0.5, "Nenhum valor nulo encontrado neste subconjunto de colunas.",
                ha="center", va="center", fontsize=12, color="#28a745")
        ax.axis("off")
        
    plt.tight_layout()
    plt.show()
    
    # 2. Matriz Espectral de Nulos (missingno)
    print("\n--- Matriz de Localização Visual de Nulos (missingno.matrix) ---")
    msno.matrix(df_subset, figsize=figsize_matrix, sparkline=True, fontsize=10, color=(0.18, 0.38, 0.58))
    plt.show()

#### 6.1 Resumo Geral de Qualidade de Dados (Todas as Colunas)

In [ ]:
df_qualidade = resumo_qualidade_dados(df_raw)
df_qualidade

#### 6.2 Análise de Duplicidade e Integridade de Chaves

In [ ]:
dups_report = analisar_duplicidades(df_raw)
print(json.dumps(dups_report, indent=2, ensure_ascii=False))

#### 6.3 Diagnóstico Visual de Nulos (Dataset Completo)

In [ ]:
plotar_diagnostico_nulos(df_raw)

#### 6.4 Diagnóstico Detalhado por Grupo Semântico

In [ ]:
# Diagnóstico do grupo de Identificação
resumo_qualidade_dados(df_raw, colunas=COLS_ID)

In [ ]:
# Diagnóstico de Situação do Atleta e Mercado
resumo_qualidade_dados(df_raw, colunas=COLS_SITUACAO + COLS_MERCADO)

In [ ]:
# Diagnóstico de Contexto e Target
resumo_qualidade_dados(df_raw, colunas=COLS_CONTEXTO + COLS_TARGET)

In [ ]:
# Diagnóstico de Scouts
resumo_qualidade_dados(df_raw, colunas=COLS_SCOUTS)